# 01. Intake, panel, audit and features
Start with notebook 01. Use the same Drive project folder. Run one stage cell at a time and share its output for review. Never use Run all across stages.


## Connect Drive


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## Get main from GitHub into temporary Colab storage
Code is cloned from main into /content/foodprice_watch, or updated there on later runs. Drive is used only for persistent artifacts. The old Drive checkout is not pulled or changed.


In [ ]:
%%bash
set -e
if [ ! -d /content/foodprice_watch/.git ]; then
    git clone --single-branch --branch main https://github.com/mrchowdhury587/foodprice_watch.git /content/foodprice_watch
else
    git -C /content/foodprice_watch pull --ff-only origin main
fi


## Initialize project runtime
Persistent artifacts are saved under the Drive project folder.


Current project settings come from GitHub. The Drive driver map is the frozen S04 artifact. Only explicitly required missing files are initialized; historical reports, state and seals are never replaced.


In [ ]:
%cd /content/foodprice_watch
%env PYTHONPATH=/content/foodprice_watch/src
%env FPW_ARTIFACT_ROOT=/content/drive/MyDrive/FoodPriceWatch
from pathlib import Path
import csv, hashlib, json, shutil
ROOT = Path('/content/drive/MyDrive/FoodPriceWatch')
for folder in ['inbox/project', 'inbox/external', 'data/processed', 'outputs',
               'reports/tables', 'reports/figures', 'reports/project/tables',
               'state', 'config', 'logs', 'models']:
    (ROOT / folder).mkdir(parents=True, exist_ok=True)
# First-run initialization only; never replace existing artifacts or seals.
for name in ['state/stages.json', 'config/item_driver_map.yaml',
             'DECISIONS.md', 'PROGRESS.md', 'DEVIATIONS.md',
             'reports/project/tables/P2_audit.csv',
             'reports/project/tables/P2_audit_summary.csv']:
    if not (ROOT / name).exists():
        shutil.copy2(name, ROOT / name)
json.loads((ROOT / 'state/stages.json').read_text())
CODE_COMMIT = !git rev-parse HEAD
assert len(CODE_COMMIT) == 1 and len(CODE_COMMIT[0]) in (40, 64) and all(c in '0123456789abcdef' for c in CODE_COMMIT[0]), 'Cannot identify code revision'
CODE_COMMIT = CODE_COMMIT[0]
CHECKS_PASS = False
stage_ok = {}


## 3. Install runtime packages


In [ ]:
%pip install -q -r requirements-colab.txt


## 4. Startup verification
A failed check raises an error and blocks the stage cells below. Missing or changed finished outputs are reset to pending by the project checker; paste failures back before continuing.


In [ ]:
for folder in ['outputs', 'data']:
    for tmp in (ROOT / folder).rglob('*.tmp'):
        assert tmp.resolve().is_relative_to(ROOT.resolve())
        tmp.unlink()
        print('Removed interrupted-write temporary file:', tmp.relative_to(ROOT))


In [ ]:
CHECKS_PASS = False
stage_ok.clear()
!python -m fpw.check
CHECKS_PASS = get_ipython().user_ns['_exit_code'] == 0
assert CHECKS_PASS, 'Startup check failed; fix it before running stages'


In [ ]:
def summarize_unit(sid):
    assert stage_ok.get(sid), 'Stage command did not succeed in this session'
    ledger = json.loads((ROOT / 'state/stages.json').read_text())
    stage = next(s for s in ledger['stages'] if s['id'] == sid)
    assert stage['status'] == 'done', 'STOP: stage was not finalized'
    summary = []
    for name, expected in stage['output_sha256'].items():
        path = ROOT / name
        actual = hashlib.sha256(path.read_bytes()).hexdigest()
        assert actual == expected, 'STOP: output hash mismatch: ' + name
        rows = None
        if path.suffix == '.csv':
            with path.open(newline='') as f:
                rows = sum(1 for _ in csv.DictReader(f))
        elif path.suffix == '.parquet':
            import pyarrow.parquet as pq
            rows = pq.read_metadata(path).num_rows
        summary.append(dict(file=name, rows=rows, sha256=actual))
    assert summary, 'STOP: no finalized output hashes'
    print(json.dumps(dict(unit=sid, outputs=summary, check='PASS', verification_code_commit=CODE_COMMIT), indent=2))


## 5. S00 — refresh intake
Before running, place the extended CSV `tcb_dhaka_daily_prices_2021_2026_v2.csv` in `inbox/project/`.
Existing original project files must be in `data/raw/project/` or `inbox/project/`.
This refreshes the intake manifest and its finalized hashes. Paste the output back and stop.


In [ ]:
assert CHECKS_PASS, 'Run and pass startup checks first'
stage_ok['S00'] = False
!python src/stages/s00_intake.py && python src/stages/finalize.py S00
stage_ok['S00'] = get_ipython().user_ns['_exit_code'] == 0
assert stage_ok['S00'], 'Stage failed; stop and review output'


In [ ]:
summarize_unit('S00')


## 6. S22 — extended panel
Run after S00 output is reviewed. The earlier S01 panel must already exist on Drive for the equality checks. Audit tables from the work-PC audit must be available. Missing prerequisites require agent review; do not rerun the old benchmark.


In [ ]:
assert CHECKS_PASS, 'Run and pass startup checks first'
stage_ok['S22'] = False
!python -m fpw.runner S22
stage_ok['S22'] = get_ipython().user_ns['_exit_code'] == 0
assert stage_ok['S22'], 'Stage failed; stop and review output'


In [ ]:
summarize_unit('S22')


## 7. S23 — kept features
Run after S22 output is reviewed. Requires the earlier driver artifacts and feature table for comparison. Paste the output back and stop.


In [ ]:
assert CHECKS_PASS, 'Run and pass startup checks first'
stage_ok['S23'] = False
!python -m fpw.runner S23
stage_ok['S23'] = get_ipython().user_ns['_exit_code'] == 0
assert stage_ok['S23'], 'Stage failed; stop and review output'


In [ ]:
summarize_unit('S23')
